# S7-0 — one adult's full measurement set from two silhouettes (Colab)

specs/s7-0-foundation/spec.md (approved 24 Nov 2026). Uses the r4 model already on Drive; trains nothing.
The input is a BodyM **Test-B** silhouette pair (varied photos, closer to real use: D-028). Turning a phone
photo into a silhouette is S7a (not built yet).

Women get 24 measurements, men 18. Today only the photo model is installed, so ANSUR (S5a) and
geometry (S5b) rows show `not_available`. Each value shows its source, quality and Test-B error.


## 1. Drive and data (same as notebook 03)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code

In [ ]:
BRANCH = "main"
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e /content/repo
!git -C /content/repo log -1 --format="commit %h  %s"
GEO, CACHE = f"{DRIVE}/geo", f"{DRIVE}/cache"
RUN = f"{DRIVE}/runs/r4"

# A package installed in a running Colab session is only importable after a restart;
# pointing Python at the code directly avoids that (cell 4 imports it).
import sys
sys.path.insert(0, "/content/repo/src")


## 3. Make sure r4 has its Test-B report (the errors shown come from it)
Skipped if `eval_report.json` already holds a Test-B section. Otherwise scores r4 on Test-B once (a few minutes).


In [ ]:
import json
import os

rep = f"{RUN}/eval_report.json"
if os.path.exists(rep) and "testB" in json.load(open(rep)):
    print("Test-B report found:", rep)
else:
    !anthro evaluate --data {DATA_DIR} --ckpt {RUN}/best.pt --cache {CACHE} --geo-cache {GEO} --test testB


## 4. Pick one woman and one man from Test-B
Only photo ids, sex, height and weight are printed (no images).


In [ ]:
from anthro.data import load_split
tb = load_split(DATA_DIR, "testB")
woman = tb[tb.gender == "female"].iloc[0]
man = tb[tb.gender == "male"].iloc[0]
for p in (woman, man):
    print(p.photo_id, p.gender, round(p.height_cm, 1), "cm", round(p.weight_kg, 1), "kg")


def mask(p, side):
    return f"{DATA_DIR}/testB/{'mask_left' if side else 'mask'}/{p.photo_id}.png"


## 5. Woman, with her weight
**Paste back:** everything this cell prints.


In [ ]:
!anthro predict-silhouettes --front {mask(woman, 0)} --side {mask(woman, 1)} --height {woman.height_cm:.1f} --weight {woman.weight_kg:.1f} --sex female --age-range 25-34 --ckpt {RUN}/best.pt --out /content/woman.json


## 6. Man, weight left out (typical weight with a warning, D-027)
`--data` is needed because r4 was trained before S7-0 stored the typical-weight line. BodyM has no ages, so `25-34` is a placeholder.
**Paste back:** everything this cell prints.


In [ ]:
!anthro predict-silhouettes --front {mask(man, 0)} --side {mask(man, 1)} --height {man.height_cm:.1f} --sex male --age-range 25-34 --ckpt {RUN}/best.pt --data {DATA_DIR} --out /content/man.json


## 7. The saved result (the `anthro-output/1` JSON the app or a tailor screen would read)

In [ ]:
print(open("/content/woman.json").read()[:1500])

## Before committing this notebook
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs (POLICY 6).
